### Modeling Earth's Climate (Pfalzgraff, Eklof & Neshyba, 2026)
# Scheduled Flows with Long-term Emissions

## Overview
The idea of this module is to create an emissions scenario -- a _schedule_ -- that describes how much carbon humans have released to the atmosphere in the past, and that makes projections about future emissions. We've done this before (ScheduledFlows), but the scenario we made there decarbonizes all the way to zero. Here we'll let emissions level off instead at some *long-term* value after the peak.

## Adding long-term emissions
Up to the year of peak emissions, nothing changes: we use Eqs. 1 and 2 of ScheduledFlows exactly as before. After the peak, we replace the scenario with

$$
\epsilon_{LTE}(t) = \epsilon_{LT} + (\epsilon_{peak} - \epsilon_{LT}) \ e^{-(t - t_{peak})^2 / t_{decarb}^2} \ \ \ \ (3)
$$

where $\epsilon_{LT}$ is the long-term emission rate we want, and $\epsilon_{peak}$ is the peak value of the original scenario. At the peak itself, Eq. 3 gives $\epsilon_{peak}$, so the two pieces join smoothly. Long after the peak, the exponential dies away and emissions settle at $\epsilon_{LT}$. The decarbonization time, $t_{decarb}$, sets how quickly that happens.

To build this in Python, we'll loop over the time array, keeping the original value of the scenario up to the peak and applying Eq. 3 after it -- the same kind of loop-with-an-if that we used in IfAndFor.

## Learning goals

1. I can explain what each term in Eq. 3 does.
1. I can extend an emissions scenario with long-term emissions, using a loop over time.
1. I can save the scenario, with its metadata, to a file for later use.

In [ ]:
%pip install -q ipympl

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

### Building an emissions scenario with long-term emissions
In the cell below, we calculate and plot an emissions scenario with long-term emissions, $\epsilon_{LTE}(t)$. There are three steps:

1. Calculate the scenario without long-term emissions, using Eqs. 1 and 2. This is exactly the code from ScheduledFlows.
1. Find the peak. Python's np.argmax gives the *index* of the largest value in an array, so t[ipeak] is the year of the peak and myeps[ipeak] is the peak emission rate.
1. Loop over the time array. Up to the peak, keep the original value; after the peak, use Eq. 3. Each value gets appended to a new array, myeps_LTE.

The graph shows both scenarios, so you can see what the long-term emissions did.

In [ ]:
# Parameters (the same as in ScheduledFlows, plus the long-term emission rate)
t_start = 1750
t_stop = 2150
nsteps = 600
t = np.linspace(t_start,t_stop,nsteps)
t_peak = 2060
t_decarb = 15
k = 0.0166
t_0 = 2003
eps_0 = 9
epslongterm = 10

# The scenario without long-term emissions (Eqs. 1 and 2, exactly as in ScheduledFlows)
tp1 = t_peak + (t_decarb/3)*np.log(3/(k*t_decarb)-1)
myeps = np.exp(k*t) * eps_0*np.exp(-k*t_0) * np.exp(3/t_decarb*(tp1-t)) / (1+np.exp(3/t_decarb*(tp1-t)))

# Find the peak (np.argmax gives the index of the largest value in an array)
ipeak = np.argmax(myeps)
eps_peak = myeps[ipeak]
print('Emissions peak at', round(eps_peak,2), 'GtC/year in the year', round(t[ipeak],1))

# Loop over time: keep the original scenario up to the peak, and use Eq. 3 after it
myeps_LTE = np.empty(0)
for i in range(len(t)):
    if t[i] <= t[ipeak]:
        thiseps = myeps[i]
    if t[i] > t[ipeak]:
        thiseps = epslongterm + (eps_peak - epslongterm)*np.exp(-((t[i]-t[ipeak])/t_decarb)**2)
    myeps_LTE = np.append(myeps_LTE, thiseps)

# Plot both scenarios, to see what the long-term emissions did
plt.figure()
plt.plot(t,myeps,label='without long-term emissions')
plt.plot(t,myeps_LTE,label='with long-term emissions')
plt.grid(True)
plt.title('Emission scenario (GtC/year)')
plt.xlabel('year')
plt.ylabel('GtC/year')
plt.legend()

<!-- BEGIN QUESTION -->

### Your turn
In the cell below, create another emissions scenario with long-term emissions. You can copy the code from the cell above; the parameters can be whatever you like, but you should make at least two changes:

- give it a lower long-term emissions amount (since 10 GtC/year is pretty high)
- give it an earlier peak emissions year (since 2060 is pretty late)

Then plot myeps_LTE as a function of time (t) to make sure it looks OK.

_Points:_ 3

In [ ]:
...

<!-- END QUESTION -->

### Setting up your data with metadata for storage
In the cell below we create the dictionary, as we did in ScheduledFlows. If you named your arrays "t" and "myeps_LTE", you'll be able to use this as-is.

In [ ]:
# Create an empty dictionary
epsdictionary = dict()

# Create an empty dataframe
epsdf = pd.DataFrame()

# Insert the time and emissions columns into the dataframe
epsdf.insert(loc=0, column='time', value=t)
epsdf.insert(loc=1, column='emissions', value=myeps_LTE)

# Add the dataframe to the dictionary
epsdictionary['dataframe']=epsdf

# Add metadata
epsdictionary['emission units'] = 'GtC/year'
epsdictionary['t_0'] = t_0
epsdictionary['eps_0'] = eps_0
epsdictionary['t_peak'] = t_peak
epsdictionary['t_decarb'] = t_decarb
epsdictionary['k'] = k
epsdictionary['epslongterm'] = epslongterm

# Report the contents of the dictionary
display(epsdictionary)

<!-- BEGIN QUESTION -->

### Saving your emissions scenario
Use the cell below to save your emissions scenario (the entire dictionary -- data and metadata) as a file, using pickle as we did in ScheduledFlows. Here's some sample code -- although you might want to modify the filename to something more meaningful, especially since we'll be eventually saving multiple scenarios.

    # Decide on a name for the file, and then save to that file
    filename = 'Peaks_in_2040_LTE.pkl'
    with open(filename, 'wb') as file:
        pickle.dump(epsdictionary, file)

After you run the cell, the file should appear in the file browser on the left, in the same folder as this notebook.

_Points:_ 4

In [ ]:
# Assign a name for the file, and save it
...

<!-- END QUESTION -->

### Double-checking
It's often nice to double-check that you really did save what you thought you did. The cell below loads (reads) the file you just saved, displays what it found, and plots the emissions it contains.

In [ ]:
# This shows what I thought I saved
print('What I thought I saved:')
display(epsdictionary)

# This loads the file back in ('rb' means open for reading), and shows what was actually saved
with open(filename, 'rb') as file:
    epsdictionary_fromfile = pickle.load(file)
print('What was actually saved:')
display(epsdictionary_fromfile)

# This pulls the time and emissions back out of the dataframe, and plots them
epsdf_fromfile = epsdictionary_fromfile['dataframe']
time = np.array(epsdf_fromfile['time'])
eps = np.array(epsdf_fromfile['emissions'])
plt.figure()
plt.plot(time, eps, 'k')
plt.xlabel('year')
plt.ylabel(epsdictionary_fromfile['emission units'])
plt.title(filename)
plt.grid(True)

### Downloading and submitting your scenario
As in ScheduledFlows, submit two files to Gradescope: this notebook and your .pkl file. To download the .pkl, right-click on it in the file browser and choose "Download". Keep both files somewhere you can find them: this scenario will be used in several later modules. If the file ever goes missing, download it from your Gradescope submission, or re-run this notebook, which recreates it.

### Validating and finishing up
Assuming all this has gone smoothly, don't forget to do a Kernel/Restart & Run All, run the whole notebook, and make sure there aren't any errors. Then download the notebook and your .pkl file, and submit both to Gradescope.